In [ ]:
# Create project folder structure
!mkdir -p project/agents project/tools project/memory project/core

In [ ]:
%%writefile project/agents/planner.py
from typing import List


class Planner:
    """Simple planner that breaks a user request into sub-tasks."""

    def plan(self, user_input: str) -> List[str]:
        """Return a list of sub-tasks for the worker agents."""
        cleaned = user_input.strip()
        if not cleaned:
            return []
        # Extremely simple heuristic: one sub-task per sentence.
        sentences = [s.strip() for s in cleaned.replace("?", ".").split(".") if s.strip()]
        if not sentences:
            sentences = [cleaned]
        subtasks = [f"Handle subtask: {s}" for s in sentences]
        return subtasks

In [ ]:
%%writefile project/agents/worker.py
from typing import Any, Dict, List

from project.tools.tools import ToolRegistry
from project.memory.session_memory import SessionMemory


class Worker:
    """Worker agent that executes sub-tasks, optionally using tools and memory."""

    def __init__(self, tools: ToolRegistry, memory: SessionMemory):
        self.tools = tools
        self.memory = memory

    def execute(self, subtask: str) -> Dict[str, Any]:
        """Execute a subtask and return a result dictionary."""
        # For this demo, we just echo the subtask and show available tools.
        used_tools: List[str] = list(self.tools.tools.keys())
        history_length = len(self.memory.get_history())
        result_text = (
            f"Worker executed: '{subtask}'.\n"
            f"Tools available: {used_tools}.\n"
            f"Messages in memory: {history_length}."
        )
        self.memory.add_message("worker", result_text)
        return {
            "subtask": subtask,
            "result": result_text,
            "used_tools": used_tools,
        }

In [ ]:
%%writefile project/agents/evaluator.py
from typing import Any, Dict, List


class Evaluator:
    """Evaluator agent that synthesizes worker outputs into a final response."""

    def evaluate(self, user_input: str, plan: List[str], worker_results: List[Dict[str, Any]]) -> str:
        lines: List[str] = [
            "Multi-agent evaluation complete.",
            f"Original request: {user_input}",
            "Plan:",
        ]
        for i, step in enumerate(plan, start=1):
            lines.append(f"  {i}. {step}")
        lines.append("\nWorker results:")
        for i, r in enumerate(worker_results, start=1):
            lines.append(f"  Step {i}: {r.get('result', '')}")
        lines.append("\nSummary: This is a simple demo response generated by the Evaluator agent.")
        return "\n".join(lines)

In [ ]:
%%writefile project/tools/tools.py
from typing import Any, Callable, Dict


class ToolRegistry:
    """Registry for callable tools worker agents can use."""

    def __init__(self) -> None:
        self.tools: Dict[str, Callable[..., Any]] = {}
        self._register_builtin_tools()

    def _register_builtin_tools(self) -> None:
        self.register_tool("echo", self.echo)
        self.register_tool("word_count", self.word_count)

    def register_tool(self, name: str, func: Callable[..., Any]) -> None:
        self.tools[name] = func

    def call(self, name: str, *args: Any, **kwargs: Any) -> Any:
        tool = self.tools.get(name)
        if not tool:
            raise ValueError(f"Tool '{name}' is not registered.")
        return tool(*args, **kwargs)

    # Example tools

    def echo(self, text: str) -> str:
        return text

    def word_count(self, text: str) -> int:
        return len(text.split())

In [ ]:
%%writefile project/memory/session_memory.py
from typing import List, Dict, Any


class SessionMemory:
    """In-memory store of the current conversation/session."""

    def __init__(self) -> None:
        self._messages: List[Dict[str, Any]] = []

    def add_message(self, role: str, content: str) -> None:
        self._messages.append({"role": role, "content": content})

    def get_history(self) -> List[Dict[str, Any]]:
        return list(self._messages)

    def clear(self) -> None:
        self._messages.clear()

In [ ]:
%%writefile project/core/context_engineering.py
from typing import List, Dict, Any


def build_system_context(user_input: str, plan: List[str], history: List[Dict[str, Any]]) -> str:
    """Create a synthetic 'system prompt' style string for the agents."""
    lines: List[str] = ["You are a simple multi-agent system demo."]
    lines.append(f"User input: {user_input}")
    lines.append("Planned steps:")
    for i, step in enumerate(plan, start=1):
        lines.append(f"  {i}. {step}")
    lines.append("Conversation history (most recent last):")
    for m in history[-5:]:
        lines.append(f"  [{m['role']}] {m['content']}")
    return "\n".join(lines)

In [ ]:
%%writefile project/core/observability.py
from typing import Any, Dict, List


class Observability:
    """Very lightweight observability helper to track events inside the agent."""

    def __init__(self) -> None:
        self.events: List[Dict[str, Any]] = []

    def log_event(self, event_type: str, payload: Dict[str, Any]) -> None:
        entry = {"type": event_type, "payload": payload}
        self.events.append(entry)

    def dump_events(self) -> List[Dict[str, Any]]:
        return list(self.events)

In [ ]:
%%writefile project/core/a2a_protocol.py
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional


@dataclass
class AgentMessage:
    """Basic in-memory representation of a message between agents."""

    sender: str
    recipient: str
    content: str
    metadata: Dict[str, Any] = field(default_factory=dict)


@dataclass
class ConversationTrace:
    """A simple trace of agent-to-agent messages for debugging/inspection."""

    messages: List[AgentMessage] = field(default_factory=list)

    def add(self, message: AgentMessage) -> None:
        self.messages.append(message)

    def to_dict(self) -> List[Dict[str, Any]]:
        return [
            {
                "sender": m.sender,
                "recipient": m.recipient,
                "content": m.content,
                "metadata": m.metadata,
            }
            for m in self.messages
        ]

In [ ]:
%%writefile project/main_agent.py
from typing import Any, Dict, List

from project.agents.planner import Planner
from project.agents.worker import Worker
from project.agents.evaluator import Evaluator
from project.tools.tools import ToolRegistry
from project.memory.session_memory import SessionMemory
from project.core.context_engineering import build_system_context
from project.core.observability import Observability
from project.core.a2a_protocol import AgentMessage, ConversationTrace


class MainAgent:
    """Top-level orchestrator for the multi-agent system."""

    def __init__(self) -> None:
        self.memory = SessionMemory()
        self.tools = ToolRegistry()
        self.planner = Planner()
        self.evaluator = Evaluator()
        self.observability = Observability()
        self.trace = ConversationTrace()

    def handle_message(self, user_input: str) -> Dict[str, Any]:
        # Log and store the user message
        self.memory.add_message("user", user_input)
        self.observability.log_event("user_message", {"content": user_input})

        # PLAN
        plan = self.planner.plan(user_input)
        self.observability.log_event("plan_created", {"plan": plan})
        self.trace.add(
            AgentMessage(
                sender="planner",
                recipient="worker",
                content="; ".join(plan),
                metadata={"stage": "plan"},
            )
        )

        # WORKER EXECUTION
        worker = Worker(self.tools, self.memory)
        worker_results: List[Dict[str, Any]] = []
        for step in plan:
            result = worker.execute(step)
            worker_results.append(result)
            self.observability.log_event("worker_step", result)
            self.trace.add(
                AgentMessage(
                    sender="worker",
                    recipient="evaluator",
                    content=result.get("result", ""),
                    metadata={"subtask": step},
                )
            )

        # CONTEXT ENGINEERING
        system_context = build_system_context(user_input, plan, self.memory.get_history())
        self.observability.log_event("system_context_built", {"context_preview": system_context[:200]})

        # EVALUATION
        final_response = self.evaluator.evaluate(user_input, plan, worker_results)
        self.memory.add_message("assistant", final_response)
        self.observability.log_event("final_response", {"response_preview": final_response[:200]})
        self.trace.add(
            AgentMessage(
                sender="evaluator",
                recipient="user",
                content=final_response,
                metadata={"stage": "final"},
            )
        )

        return {
            "response": final_response,
            "plan": plan,
            "worker_results": worker_results,
            "system_context": system_context,
            "events": self.observability.dump_events(),
            "trace": self.trace.to_dict(),
        }


def run_agent(user_input: str):
    agent = MainAgent()
    result = agent.handle_message(user_input)
    return result["response"]

In [ ]:
%%writefile project/app.py
from project.main_agent import run_agent


def main() -> None:
    print("Multi-agent demo app. Type 'quit' to exit.\n")
    while True:
        try:
            user_input = input("You: ")
        except EOFError:
            break
        if user_input.lower().strip() in {"quit", "exit"}:
            print("Exiting.")
            break
        response = run_agent(user_input)
        print("Agent:\n" + response + "\n")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile project/run_demo.py
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
from project.main_agent import run_agent


if __name__ == "__main__":
    print(run_agent("Hello! This is a demo."))

In [ ]:
%%writefile project/requirements.txt
# No external dependencies are required for this simple multi-agent demo.
# You can install additional libraries here if you extend the project.

In [ ]:
from project.main_agent import run_agent
print(run_agent("Hello!"))

In [ ]:
!zip -r project.zip project